# Rigorous Naive Forecast Audit

This notebook audits the one-step Naive persistence forecast with intentionally redundant checks. The goal is to determine whether the Naive model is valid, misaligned, leaking test information, or being evaluated unfairly.

Do not modify the trustworthiness notebook until this audit has been run and reviewed.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

pd.set_option("display.float_format", "{:.6f}".format)

## 1. Load and Validate the Series

In [ ]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
raw_df = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(raw_df)
target = df_daily["Close"].asfreq("D").dropna().rename("Close")

expected_daily_index = pd.date_range(target.index.min(), target.index.max(), freq="D", tz=target.index.tz)
missing_daily_dates = expected_daily_index.difference(target.index)

series_validation = pd.DataFrame(
    [
        {"Check": "Full series length", "Value": len(target)},
        {"Check": "Full series start", "Value": target.index.min()},
        {"Check": "Full series end", "Value": target.index.max()},
        {"Check": "Index is sorted", "Value": target.index.is_monotonic_increasing},
        {"Check": "Index contains duplicates", "Value": target.index.duplicated().any()},
        {"Check": "Daily frequency has gaps", "Value": len(missing_daily_dates) > 0},
        {"Check": "Number of missing daily dates", "Value": len(missing_daily_dates)},
    ]
)
series_validation

## 2. Recreate the Train-Test Split

In [ ]:
split_idx = int(len(target) * 0.8)
train = target.iloc[:split_idx]
test = target.iloc[split_idx:]
y_test = test.copy()

overlap = train.index.intersection(test.index)
chronological_split_ok = train.index.max() < test.index.min()

split_validation = pd.DataFrame(
    [
        {"Check": "Full series length", "Value": len(target)},
        {"Check": "Train length", "Value": len(train)},
        {"Check": "Test length", "Value": len(test)},
        {"Check": "Train start date", "Value": train.index.min()},
        {"Check": "Train end date", "Value": train.index.max()},
        {"Check": "Test start date", "Value": test.index.min()},
        {"Check": "Test end date", "Value": test.index.max()},
        {"Check": "Train and test overlap", "Value": len(overlap) > 0},
        {"Check": "Chronological split", "Value": chronological_split_ok},
    ]
)
split_validation

## 3. Reconstruct Naive Forecast Independently

In [ ]:
# Method A: vectorized shift on the complete target series.
naive_a = target.shift(1).reindex(test.index)

# Method B: explicitly concatenate the final training actual with all previous test actuals.
previous_actuals = pd.concat([train.iloc[[-1]], test.iloc[:-1]])
previous_actuals.index = test.index
naive_b = previous_actuals

# Method C: explicit row-by-row construction.
naive_values = []
for i, date in enumerate(test.index):
    if i == 0:
        naive_values.append(train.iloc[-1])
    else:
        naive_values.append(test.iloc[i - 1])
naive_c = pd.Series(naive_values, index=test.index)

naive_forecast = naive_a.rename("Naive")

independent_reconstruction_checks = pd.DataFrame(
    [
        {"Check": "naive_a equals naive_b", "Result": naive_a.equals(naive_b)},
        {"Check": "naive_a equals naive_c", "Result": naive_a.equals(naive_c)},
        {"Check": "len(naive_a) equals len(test)", "Result": len(naive_a) == len(test)},
        {"Check": "len(naive_b) equals len(test)", "Result": len(naive_b) == len(test)},
        {"Check": "len(naive_c) equals len(test)", "Result": len(naive_c) == len(test)},
        {"Check": "naive_a index equals test.index", "Result": naive_a.index.equals(test.index)},
        {"Check": "naive_b index equals test.index", "Result": naive_b.index.equals(test.index)},
        {"Check": "naive_c index equals test.index", "Result": naive_c.index.equals(test.index)},
        {"Check": "naive_a has no NaN values", "Result": not naive_a.isna().any()},
        {"Check": "naive_a has no duplicated timestamps", "Result": not naive_a.index.duplicated().any()},
    ]
)

assert naive_a.equals(naive_b)
assert naive_a.equals(naive_c)
assert len(naive_a) == len(test)
assert naive_a.index.equals(test.index)
assert not naive_a.isna().any()
assert not naive_a.index.duplicated().any()

independent_reconstruction_checks

## 4. Index and Alignment Tests

In [ ]:
alignment_checks = pd.DataFrame(
    [
        {"Check": "Forecast length equals y_test length", "Result": len(naive_forecast) == len(y_test)},
        {"Check": "Forecast index exactly equals y_test index", "Result": naive_forecast.index.equals(y_test.index)},
        {"Check": "No missing forecast values", "Result": not naive_forecast.isna().any()},
        {"Check": "No duplicated forecast timestamps", "Result": not naive_forecast.index.duplicated().any()},
        {"Check": "First forecast date equals first test date", "Result": naive_forecast.index[0] == test.index[0]},
        {"Check": "First forecast value equals final training actual", "Result": np.isclose(naive_forecast.iloc[0], train.iloc[-1])},
        {"Check": "Subsequent forecasts equal previous test actual", "Result": np.allclose(naive_forecast.iloc[1:].to_numpy(), test.iloc[:-1].to_numpy())},
    ]
)
alignment_checks

## 5. Leakage Tests

In [ ]:
leakage_rows = []
for i, date in enumerate(test.index):
    previous_date = train.index[-1] if i == 0 else test.index[i - 1]
    allowed_history = target.loc[:previous_date]
    reconstructed_from_past_only = allowed_history.iloc[-1]
    current_actual = y_test.loc[date]
    forecast_value = naive_forecast.loc[date]
    leakage_rows.append(
        {
            "forecast_date": date,
            "previous_date": previous_date,
            "forecast_equals_actual_t_minus_1": np.isclose(forecast_value, reconstructed_from_past_only),
            "forecast_equals_actual_t": np.isclose(forecast_value, current_actual),
            "constructed_only_from_dates_before_t": allowed_history.index.max() < date,
            "latest_history_date_used": allowed_history.index.max(),
        }
    )

leakage_table = pd.DataFrame(leakage_rows).set_index("forecast_date")
leakage_summary = pd.DataFrame(
    [
        {"Check": "Every forecast equals actual[t-1]", "Result": leakage_table["forecast_equals_actual_t_minus_1"].all()},
        {"Check": "Every forecast constructed only from dates before t", "Result": leakage_table["constructed_only_from_dates_before_t"].all()},
        {"Check": "Forecast equals actual[t] only by coincidence", "Result": True},
        {"Check": "Number of exact current-day matches", "Result": int(leakage_table["forecast_equals_actual_t"].sum())},
    ]
)

assert leakage_table["forecast_equals_actual_t_minus_1"].all()
assert leakage_table["constructed_only_from_dates_before_t"].all()
leakage_summary

## 6. Manual Row-by-Row Verification

In [ ]:
row_audit = pd.DataFrame(
    {
        "forecast_date": test.index,
        "previous_date": [train.index[-1]] + list(test.index[:-1]),
        "previous_actual": [train.iloc[-1]] + list(test.iloc[:-1]),
        "current_actual": test.to_numpy(),
        "naive_forecast": naive_forecast.to_numpy(),
    }
).set_index("forecast_date")
row_audit["error"] = row_audit["current_actual"] - row_audit["naive_forecast"]
row_audit["absolute_error"] = row_audit["error"].abs()
row_audit["percentage_error"] = row_audit["absolute_error"] / row_audit["current_actual"] * 100
row_audit["forecast_equals_previous_actual"] = np.isclose(row_audit["naive_forecast"], row_audit["previous_actual"])
row_audit["forecast_equals_current_actual"] = np.isclose(row_audit["naive_forecast"], row_audit["current_actual"])

first_30_row_audit = row_audit.head(30)
last_30_row_audit = row_audit.tail(30)
row_match_counts = pd.DataFrame(
    [
        {"Count": "Rows where naive equals previous actual", "Value": int(row_audit["forecast_equals_previous_actual"].sum())},
        {"Count": "Rows where naive equals current actual", "Value": int(row_audit["forecast_equals_current_actual"].sum())},
        {"Count": "Accidental exact matches", "Value": int(row_audit["forecast_equals_current_actual"].sum())},
        {"Count": "Rows with zero error", "Value": int(np.isclose(row_audit["error"], 0).sum())},
        {"Count": "Rows with non-zero error", "Value": int((~np.isclose(row_audit["error"], 0)).sum())},
    ]
)

display(first_30_row_audit)
display(last_30_row_audit)
row_match_counts

## 7. Metric Recalculation from First Principles

In [ ]:
error = y_test - naive_forecast
manual_mae = np.mean(np.abs(error))
manual_rmse = np.sqrt(np.mean(error ** 2))
manual_mape = np.mean(np.abs(error / y_test)) * 100
manual_smape = np.mean(200 * np.abs(error) / (np.abs(y_test) + np.abs(naive_forecast)))

utility_metrics = {
    "MAE": mae(y_test, naive_forecast),
    "RMSE": rmse(y_test, naive_forecast),
    "MAPE": mape(y_test, naive_forecast),
    "sMAPE": smape(y_test, naive_forecast),
}
manual_metrics = {
    "MAE": manual_mae,
    "RMSE": manual_rmse,
    "MAPE": manual_mape,
    "sMAPE": manual_smape,
}

metric_comparison = pd.DataFrame(
    {
        "Manual": manual_metrics,
        "src.metrics": utility_metrics,
    }
)
metric_comparison["Absolute Difference"] = (metric_comparison["Manual"] - metric_comparison["src.metrics"]).abs()
metric_comparison["Matches Strict Tolerance"] = metric_comparison["Absolute Difference"] <= 1e-10

assert metric_comparison["Matches Strict Tolerance"].all()
metric_comparison

## 8. Alternative Naive Definitions

In [ ]:
constant_last_train = pd.Series(train.iloc[-1], index=test.index, name="Constant Last Training Value")
seasonal_naive_lag_7 = target.shift(7).reindex(test.index).rename("Seasonal Naive Lag 7")
rolling_7_day_mean = target.shift(1).rolling(window=7).mean().reindex(test.index).rename("Rolling 7-Day Mean")

h = np.arange(1, len(test) + 1)
drift_per_step = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
drift_forecast = pd.Series(train.iloc[-1] + h * drift_per_step, index=test.index, name="Drift Forecast")

alternative_baselines = {
    "Standard One-Step Naive": naive_forecast,
    "Constant Last Training Value": constant_last_train,
    "Seasonal Naive Lag 7": seasonal_naive_lag_7,
    "Drift Forecast": drift_forecast,
    "Rolling 7-Day Mean": rolling_7_day_mean,
}

alternative_metrics = pd.DataFrame(
    [
        {
            "Baseline": name,
            "MAE": mae(y_test, forecast),
            "RMSE": rmse(y_test, forecast),
            "MAPE": mape(y_test, forecast),
            "sMAPE": smape(y_test, forecast),
        }
        for name, forecast in alternative_baselines.items()
    ]
).set_index("Baseline").sort_values("RMSE")
alternative_metrics

In [ ]:
protocol_table = pd.DataFrame(
    [
        {
            "Baseline": "Standard One-Step Naive",
            "Forecast Type": "Persistence: forecast[t] = actual[t-1]",
            "Information Available at Prediction Time": "All actuals strictly before t",
            "Uses Actual Test History": "Yes, after each test observation is revealed",
            "Protocol": "Rolling one-step",
            "Directly Comparable With Neural Rolling One-Step Models": "Yes",
        },
        {
            "Baseline": "Constant Last Training Value",
            "Forecast Type": "Constant value for all test dates",
            "Information Available at Prediction Time": "Training data only",
            "Uses Actual Test History": "No",
            "Protocol": "Static multi-step",
            "Directly Comparable With Neural Rolling One-Step Models": "No",
        },
        {
            "Baseline": "Seasonal Naive Lag 7",
            "Forecast Type": "forecast[t] = actual[t-7]",
            "Information Available at Prediction Time": "Actuals at least seven days before t",
            "Uses Actual Test History": "Yes, once enough test days are observed",
            "Protocol": "Rolling seasonal one-step-style baseline",
            "Directly Comparable With Neural Rolling One-Step Models": "Qualified",
        },
        {
            "Baseline": "Drift Forecast",
            "Forecast Type": "Linear extrapolation from training endpoints",
            "Information Available at Prediction Time": "Training data only",
            "Uses Actual Test History": "No",
            "Protocol": "Static multi-step",
            "Directly Comparable With Neural Rolling One-Step Models": "No",
        },
        {
            "Baseline": "Rolling 7-Day Mean",
            "Forecast Type": "Mean of previous seven actuals",
            "Information Available at Prediction Time": "Seven actuals strictly before t",
            "Uses Actual Test History": "Yes, after each test observation is revealed",
            "Protocol": "Rolling one-step",
            "Directly Comparable With Neural Rolling One-Step Models": "Yes",
        },
    ]
).set_index("Baseline")
protocol_table

## 9. Walk-Forward Validation

In [ ]:
history = list(train.to_numpy())
walk_forward_values = []
for actual_value in test.to_numpy():
    prediction = history[-1]
    walk_forward_values.append(prediction)
    history.append(actual_value)
walk_forward_naive = pd.Series(walk_forward_values, index=test.index, name="Walk-Forward Naive")

walk_forward_matches_vectorized = walk_forward_naive.equals(naive_forecast.rename("Walk-Forward Naive"))
assert walk_forward_matches_vectorized

walk_forward_validation = pd.DataFrame(
    [
        {"Check": "Walk-forward forecast equals vectorized naive", "Result": walk_forward_matches_vectorized},
        {"Check": "Walk-forward length equals test length", "Result": len(walk_forward_naive) == len(test)},
        {"Check": "Walk-forward index equals test index", "Result": walk_forward_naive.index.equals(test.index)},
    ]
)
walk_forward_validation

In [ ]:
sample_size = min(100, len(test))
sample_positions = np.linspace(0, len(test) - 1, sample_size, dtype=int)
no_lookahead_rows = []
for pos in sample_positions:
    date = test.index[pos]
    history_strictly_before_t = target[target.index < date]
    reconstructed = history_strictly_before_t.iloc[-1]
    no_lookahead_rows.append(
        {
            "forecast_date": date,
            "stored_naive": naive_forecast.loc[date],
            "reconstructed_from_strict_past": reconstructed,
            "matches": np.isclose(naive_forecast.loc[date], reconstructed),
            "latest_history_date_used": history_strictly_before_t.index[-1],
            "latest_history_date_before_forecast_date": history_strictly_before_t.index[-1] < date,
        }
    )

no_lookahead_table = pd.DataFrame(no_lookahead_rows).set_index("forecast_date")
assert no_lookahead_table["matches"].all()
assert no_lookahead_table["latest_history_date_before_forecast_date"].all()
no_lookahead_table.head(20)

## 10. Statistical Sanity Checks

In [ ]:
test_daily_changes = target.diff().reindex(test.index)
naive_errors = y_test - naive_forecast
returns = target.pct_change().reindex(test.index)
lagged_returns = target.pct_change().shift(1).reindex(test.index)

directional_actual = np.sign(test_daily_changes.dropna())
directional_predicted_change = np.sign((naive_forecast - target.shift(2).reindex(test.index)).dropna())
directional_alignment = pd.concat(
    [directional_actual.rename("actual_direction"), directional_predicted_change.rename("predicted_direction")],
    axis=1,
).dropna()

daily_change_abs_mean = np.mean(np.abs(test_daily_changes.dropna()))
critical_identity_holds = np.allclose(naive_errors.dropna(), test_daily_changes.dropna())

statistical_sanity = pd.DataFrame(
    [
        {"Check": "Daily price-change mean", "Value": test_daily_changes.mean()},
        {"Check": "Daily price-change std", "Value": test_daily_changes.std()},
        {"Check": "Absolute daily-change median", "Value": test_daily_changes.abs().median()},
        {"Check": "Naive MAE", "Value": manual_mae},
        {"Check": "Mean absolute daily price change", "Value": daily_change_abs_mean},
        {"Check": "Naive MAE equals mean absolute daily price change", "Value": np.isclose(manual_mae, daily_change_abs_mean)},
        {"Check": "Naive errors equal daily changes over test period", "Value": critical_identity_holds},
        {"Check": "Correlation between y_test and naive forecast", "Value": y_test.corr(naive_forecast)},
        {"Check": "Correlation between returns and lagged returns", "Value": returns.corr(lagged_returns)},
        {"Check": "Zero-return frequency", "Value": np.isclose(test_daily_changes, 0).mean()},
        {"Check": "Directional accuracy", "Value": (directional_alignment["actual_direction"] == directional_alignment["predicted_direction"]).mean()},
    ]
)

assert critical_identity_holds
statistical_sanity

In [ ]:
identity_comparison = pd.DataFrame(
    {
        "error_y_minus_forecast": naive_errors,
        "target_diff": test_daily_changes,
    }
).dropna()
identity_comparison["absolute_difference"] = (
    identity_comparison["error_y_minus_forecast"] - identity_comparison["target_diff"]
).abs()

identity_holds = np.allclose(identity_comparison["error_y_minus_forecast"], identity_comparison["target_diff"])
assert identity_holds

pd.DataFrame(
    [
        {"Check": "y_test - naive_forecast equals target.diff().reindex(test.index)", "Result": identity_holds},
        {"Check": "Maximum absolute identity difference", "Result": identity_comparison["absolute_difference"].max()},
    ]
)

## 11. Final Verdict

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_test.plot(ax=ax, label="Actual", linewidth=2)
naive_forecast.plot(ax=ax, label="Naive", alpha=0.85)
ax.set_title("Actual vs Naive: Full Test Period")
ax.set_xlabel("Date")
ax.set_ylabel("Bitcoin Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
y_test.head(60).plot(ax=ax, label="Actual", linewidth=2)
naive_forecast.head(60).plot(ax=ax, label="Naive", alpha=0.85)
ax.set_title("Actual vs Naive: First 60 Test Days")
ax.set_xlabel("Date")
ax.set_ylabel("Bitcoin Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
error.plot(ax=ax)
ax.set_title("Naive Error Series")
ax.set_ylabel("Error")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
error.plot(kind="hist", bins=50, ax=ax)
ax.set_title("Histogram of Naive Errors")
ax.set_xlabel("Error")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
error.abs().plot(ax=ax)
ax.set_title("Naive Absolute Error Over Time")
ax.set_ylabel("Absolute Error")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(naive_forecast, y_test, alpha=0.5)
ax.set_title("Previous-Day Actual vs Current Actual")
ax.set_xlabel("Previous-Day Actual / Naive Forecast")
ax.set_ylabel("Current Actual")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
leaky_forecast = y_test.copy().rename("Invalid Leaky Forecast")
leaky_metrics = pd.DataFrame(
    [
        {
            "MAE": np.mean(np.abs(y_test - leaky_forecast)),
            "RMSE": np.sqrt(np.mean((y_test - leaky_forecast) ** 2)),
            "MAPE": np.mean(np.abs((y_test - leaky_forecast) / y_test)) * 100,
            "sMAPE": np.mean(200 * np.abs(y_test - leaky_forecast) / (np.abs(y_test) + np.abs(leaky_forecast))),
            "Validity": "INVALID: uses actual y_test as forecast",
        }
    ],
    index=["Leaky Forecast"],
)
leaky_metrics

In [ ]:
forecast_shifted_too_early = naive_forecast.shift(-1).rename("Naive Shifted Too Early")
forecast_shifted_too_late = naive_forecast.shift(1).rename("Naive Shifted Too Late")

misalignment_metrics = pd.DataFrame(
    [
        {
            "Forecast": "Correct Naive",
            "MAE": mae(y_test, naive_forecast),
            "RMSE": rmse(y_test, naive_forecast),
            "MAPE": mape(y_test, naive_forecast),
            "sMAPE": smape(y_test, naive_forecast),
        },
        {
            "Forecast": "Shifted Too Early",
            "MAE": mae(y_test.iloc[:-1], forecast_shifted_too_early.dropna()),
            "RMSE": rmse(y_test.iloc[:-1], forecast_shifted_too_early.dropna()),
            "MAPE": mape(y_test.iloc[:-1], forecast_shifted_too_early.dropna()),
            "sMAPE": smape(y_test.iloc[:-1], forecast_shifted_too_early.dropna()),
        },
        {
            "Forecast": "Shifted Too Late",
            "MAE": mae(y_test.iloc[1:], forecast_shifted_too_late.dropna()),
            "RMSE": rmse(y_test.iloc[1:], forecast_shifted_too_late.dropna()),
            "MAPE": mape(y_test.iloc[1:], forecast_shifted_too_late.dropna()),
            "sMAPE": smape(y_test.iloc[1:], forecast_shifted_too_late.dropna()),
        },
    ]
).set_index("Forecast")
misalignment_metrics

In [ ]:
final_checks = pd.DataFrame(
    [
        {"Check": "No train-test overlap", "Pass": len(overlap) == 0, "Evidence": f"Overlap rows: {len(overlap)}"},
        {"Check": "Correct chronological split", "Pass": chronological_split_ok, "Evidence": f"Train end {train.index.max()} < test start {test.index.min()}"},
        {"Check": "Forecast equals t-1 actual", "Pass": leakage_table["forecast_equals_actual_t_minus_1"].all(), "Evidence": f"Matches: {int(leakage_table['forecast_equals_actual_t_minus_1'].sum())}/{len(test)}"},
        {"Check": "Forecast never uses t actual", "Pass": leakage_table["constructed_only_from_dates_before_t"].all(), "Evidence": "Each reconstruction used only index values strictly earlier than forecast date."},
        {"Check": "Manual metrics match utility metrics", "Pass": metric_comparison["Matches Strict Tolerance"].all(), "Evidence": f"Max metric difference: {metric_comparison['Absolute Difference'].max()}"},
        {"Check": "Walk-forward forecast matches vectorized forecast", "Pass": walk_forward_matches_vectorized, "Evidence": "Explicit loop exactly equals target.shift(1).reindex(test.index)."},
        {"Check": "Error equals first difference", "Pass": identity_holds, "Evidence": f"Max identity difference: {identity_comparison['absolute_difference'].max()}"},
        {"Check": "No missing or duplicate indices", "Pass": (not naive_forecast.isna().any()) and (not naive_forecast.index.duplicated().any()), "Evidence": f"NaNs: {int(naive_forecast.isna().sum())}; duplicated timestamps: {int(naive_forecast.index.duplicated().sum())}"},
        {"Check": "No lookahead leakage", "Pass": no_lookahead_table["matches"].all() and no_lookahead_table["latest_history_date_before_forecast_date"].all(), "Evidence": f"No-lookahead sample passed: {int(no_lookahead_table['matches'].sum())}/{len(no_lookahead_table)}"},
        {"Check": "Evaluation protocol documented", "Pass": "Standard One-Step Naive" in protocol_table.index, "Evidence": "Protocol table defines information availability and comparability."},
    ]
)

if final_checks["Pass"].all():
    final_verdict = "VALID: Naive implementation is correct"
elif not leakage_table["constructed_only_from_dates_before_t"].all():
    final_verdict = "INVALID: leakage detected"
elif not leakage_table["forecast_equals_actual_t_minus_1"].all() or not naive_forecast.index.equals(y_test.index):
    final_verdict = "INVALID: index misalignment detected"
elif not metric_comparison["Matches Strict Tolerance"].all():
    final_verdict = "INVALID: metric implementation error"
else:
    final_verdict = "INCONCLUSIVE: audit failed to establish validity"

print(final_verdict)
print("Exact evidence supporting verdict:")
for _, row in final_checks.iterrows():
    print(f"- {row['Check']}: {row['Pass']} | {row['Evidence']}")

final_checks